In [249]:
import pandas as pd
from scipy.stats import f
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score

In [250]:
# Lendo o conjunto de dados
df = pd.read_excel('../data/online-shopping.xlsx')

In [251]:
# Drop colunas desnecessárias
df = df.drop(columns=['ID', 'Start time', 'Completion time', 'Email', 'Name', 'Last modified time'])

# Drop valores nulos
df = df.dropna()

# Transformar os nomes das plataformas para minúsculo e tirar espaços extras
df.iloc[:, 7] = df.iloc[:, 7].str.lower()
df.iloc[:, 7] = df.iloc[:, 7].str.strip()

df.head()

,Com que frequência você faz compras online?,Qual é a sua faixa etária?,Gênero,Qual é seu nível de satisfação com a interface do usuário de sites/aplicativos de compras online?,Quão satisfeito você está com a variedade de produtos disponíveis online?,"Numa escala de 1 a 5, quão satisfeito você está com os preços dos produtos online?",Com que frequência você encontra problemas técnicos ao fazer compras online?,Qual plataforma de compras online você prefere?,Qual é a probabilidade de você recomendar compras online a um amigo ou colega?,"Que fatores influenciam a sua decisão de fazer compras online? (Selecione todas as opções aplicáveis: Conveniência, Preço, Seleção de Produtos, Avaliações, Reputação da Marca, Descontos/Promoções,...","Em média, quanto você gasta em compras online?"
0,1 vez por mês,18-24 anos,Masculino,5,5,5,Raramente,amazon,5,Conveniência;Avaliações;Desconto/Promoções;Preço;,300
1,1 vez a cada 2 à 6 meses,18-24 anos,Masculino,4,5,4,Raramente,mercado livre,5,Preço;,100
2,1 vez por mês,18-24 anos,Masculino,5,5,5,Às vezes,kabum,5,Preço;,200
3,1 vez por mês,18-24 anos,Masculino,4,4,4,Raramente,amazon,5,Conveniência;Preço;Avaliações;Desconto/Promoções;,200
4,1 vez a cada 6 à 12 meses,18-24 anos,Masculino,3,4,3,Raramente,amazon,4,Preço;Avaliações;Reputação da Marca;,200


In [252]:
# Dividindo o conjunto de dados em um conjunto de treino (train_set) e um conjunto de teste (test_set):
train_set, test_set = train_test_split(df, test_size=0.2, random_state=42)

In [253]:
# Declarando as variáveis em independentes e dependentes para o conjunto de treino e de teste

# Treino

# Variáveis independentes
X_train = train_set.drop(['Em média, quanto você gasta em compras online?'], axis=1)
X_train = pd.get_dummies(X_train, drop_first=True)
X_train = X_train.astype(int)

# Variável dependentes
y_train = train_set['Em média, quanto você gasta em compras online?']

# Teste

# Variáveis independentes
X_test = test_set.drop(['Em média, quanto você gasta em compras online?'], axis=1)
X_test = pd.get_dummies(X_test, drop_first=True)
X_test = X_test.astype(int)

# Temos que garantir que as colunas de X_test são as mesmas de X_train
X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

# Variável dependentes
y_test = test_set['Em média, quanto você gasta em compras online?']

In [254]:
# Treinando o modelo com um modelo de regressão linear
model = LinearRegression()
model.fit(X_train, y_train)

# Fazer previsões no conjunto de teste - Quais os valores de Y obtido considerando o modelo de regressão linear obtido.
y_pred = model.predict(X_test)

In [255]:
# Criando uma tabela com y_test e y_pred para comparar os resultados
result_df = pd.DataFrame({'y_test': y_test, 'y_pred': y_pred})

In [256]:
# Avaliando o modelo
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

In [257]:
print(f"Mean Squared Error: {mse}")
print(f"R-squared: {r2}")

n = len(y_test)
p = X_test.shape[1]
f_statistic = (r2/(1-r2))*((n-p-1)/p)
p_value = 1 - f.cdf(f_statistic, p, n-p-1)

print(f"F-statistic: {f_statistic}")

Mean Squared Error: 45721.12129651921
R-squared: -5.723694308311648
F-statistic: 0.718852112948861


# Regressão Linear Simples:

Em um modelo de regressão linear, **Mean Squared Error (MSE)** e **R-squared (R²)** são métricas fundamentais para avaliar o desempenho do modelo:

1. **Mean Squared Error (MSE)**:
   - Representa a média dos quadrados dos erros, ou seja, a média das diferenças quadradas entre os valores observados e os valores previstos pelo modelo.
   - Quanto menor o MSE, melhor o modelo, pois indica que as previsões estão mais próximas dos valores reais.

2. **R-squared (R²)**:
   - Também conhecido como coeficiente de determinação, mede a proporção da variabilidade total da variável dependente que é explicada pelo modelo.
   - Varia entre 0 e 1, onde valores mais próximos de 1 indicam que o modelo explica bem a variação dos dados. Um R² de 0 significa que o modelo não explica nada da variabilidade dos dados.

Em resumo, o MSE mede a precisão das previsões do modelo, enquanto o R² avalia a qualidade do ajuste do modelo aos dados.

# Respostas:
1) O modelo é significativo?
    - Não, o modelo não é significativo, pois o p-value é maior que 0.05 indicando que a hipótese nula de que os parâmetros são iguais a zero é aceita.
2) Os parametros são significativos? O que isso significa?
    - Não, os parâmetros não são significativos. Isso significa que as variáveis independentes não são capazes de explicar a variabilidade da variável dependente.
3) Qual o valor do coeficiente de determinação? O que isso significa?
    - O valor do coeficiente de determinação é ~-5.72. Um valor negativo para o coeficiente de determinação não faz sentido, pois ele deve variar entre 0 e 1. Isso pode indicar que o modelo não é adequado para explicar a variabilidade dos dados.
4) Caso exista parametros não significativos, o que vocês fariam para melhorar o modelo?
    - Caso existisse parâmetros não significativos, seria necessário realizar uma análise mais aprofundada dos dados, talvez utilizando outras variáveis ou até mesmo outras técnicas de regressão. Porém, como o modelo é significativo e os parâmetros também, não é necessário realizar nenhuma melhoria.
5) Conclusões finais
    - O modelo de regressão linear não é adequado para explicar a variabilidade dos dados, pois o coeficiente de determinação é negativo e os parâmetros não são significativos. Portanto, é necessário considerar outras abordagens ou técnicas para modelar esses dados.

# Sobre o MSE e R²:

Nosso MSE foi de aproximadamente 45 mil, muitíssimo alto, sugerindo que as previsões do modelo de regressão linear estão se desviando significativamente dos valores reais.

4 possíveis fatores que acarretaram no alto MSE são:

- Grande Escala da Variável Alvo: Se a variável alvo tiver uma grande escala (por exemplo, prever preços de casas em dólares, no nosso caso prever preço gasto em compras online), o MSE alto pode ser esperado.

- Problemas de ajuste do modelo: o modelo pode estar mal ajustado ou sobreajustado. O underfitting acontece se o modelo linear for muito simples para capturar o relacionamento subjacente. O overfitting pode ocorrer se o modelo se ajustar muito aos dados de treinamento, falhando na generalização para os dados de teste. Possivelmente o modelo esta sofrendo de overfitting pois o R² é 1.

- Qualidade dos dados: pode haver problemas com os dados, como valores discrepantes ou pontos de dados ruidosos. Porém descartamos esta ideia pois o pré-processamento foi feito.

- Seleção de recursos: os recursos usados ​​podem não ser os melhores preditores para sua variável de destino. Acreditamos que este fator possa ter contribuido muito para o alto MSE pois não fizemos uma análise exploratória dos dados. Acreditamos que as variáveis independentes não possuem uma relação linear com a variável dependente e nem entre si.

Para melhorar o modelo, poderíamos tentar:
- Normalizar a variável alvo
- Adicionar mais variáveis independentes
- Utilizar um modelo de regressão não linear
- Realizar uma análise exploratória dos dados
- Utilizar técnicas de regularização

Nosso R² é de -5,72, altamente incomum, indicando que o modelo está apresentando um desempenho muito ruim.


3 possíveis razões que acarretaram no R² negativo:

- Especificação incorreta do modelo: o modelo pode estar especificado incorretamente. A regressão linear assume uma relação linear entre as variáveis ​​independentes e dependentes. Se esta suposição for violada, o modelo pode ter um desempenho insatisfatório. Como dito para o MSE elevado, acreditamos que as variáveis independentes não possuem uma relação linear com a variável dependente e nem entre si.

- Subajuste grave: um R² negativo indica que o modelo é pior do que uma linha horizontal que representa a média da variável dependente. Isso pode ser devido ao modelo não capturar a relação entre as variáveis.

- Problemas de dados: pode haver problemas significativos com os dados, como valores discrepantes, tipos de dados incorretos ou dados altamente ruidosos o que também foi notado para o MSE elevado.

Para melhorar o modelo, poderíamos tentar:
- Verificar os dados para erros e valores discrepantes
- Explorar visualmente os relacionamentos entre as variáveis
- Experimentar modelos mais flexíveis, como regressão polinomial, árvores de decisão ou florestas aleatórias
- Usar validação cruzada para garantir que o desempenho do modelo seja consistente